# Data Cleaning & Text Processing
This notebook demonstrates text lowercase, punctuation removal, stopwords removal, lemmatization, and spelling correction.

In [ ]:
import pandas as pd
import numpy as np
import re
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
import spacy

nltk.download('stopwords')
stop_words = set(stopwords.words('french'))

# Load spaCy model for French (run `python -m spacy download fr_core_news_sm` in terminal if not installed)
try:
    nlp = spacy.load("fr_core_news_sm")
except:
    nlp = None
    print("spaCy model not found, skipping lemmatization demo...")

df = pd.read_csv('../data/dataset.csv')
print(f"Loaded dataset: {df.shape}")

## 1. Basic Cleaning
Lowercasing, punctuation, stopwords, lemmatization, missing values.

In [ ]:
# Drop missing values
df = df.dropna(subset=['avis']).reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zÀ-ÿ0-9\s]', '', text)
    # Tokenize and remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    # Lemmatization
    if nlp is not None:
        doc = nlp(" ".join(words))
        words = [token.lemma_ for token in doc]
    return " ".join(words)

# Apply to a sample to demonstrate
df_sample = df.head(10).copy()
df_sample['cleaned_text'] = df_sample['avis'].apply(clean_text)
display(df_sample[['avis', 'cleaned_text']])

## 2. Spelling Correction
Using TextBlob for spelling correction on the sample.

In [ ]:
def correct_spelling(text):
    return str(TextBlob(text).correct())

df_sample['corrected_text'] = df_sample['cleaned_text'].apply(correct_spelling)
display(df_sample[['avis', 'cleaned_text', 'corrected_text']])

# Note: The full processing takes a lot of time, so we load the pre-computed enriched dataset later on.
# df.to_csv('../data/cleaned_dataset.csv', index=False)
